# GPU Assignment I - Complete Colab Notebook

Run all cells from top to bottom. This records your actual Tesla T4 GPU and creates the required output files. The assignment asks for RTX 4090/5090, so report the actual T4 and ask your instructor whether it is acceptable.

In [ ]:

import os,subprocess,json,time,math,gc,shutil
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt,torch
OUT=Path("/content/gpu_assignment_outputs");OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(),"Enable Runtime > Change runtime type > GPU."
device=torch.device("cuda"); props=torch.cuda.get_device_properties(0)
def cmd(x):
 try:return subprocess.check_output(x,shell=True,text=True,stderr=subprocess.STDOUT)
 except Exception as e:return str(e)
gpu_name=torch.cuda.get_device_name(0); uuid=cmd("nvidia-smi --query-gpu=uuid --format=csv,noheader").strip().splitlines()[0]; driver=cmd("nvidia-smi --query-gpu=driver_version --format=csv,noheader").strip().splitlines()[0]
(OUT/"nvidia-smi-q.txt").write_text(cmd("nvidia-smi -q"))
info={"gpu_name":gpu_name,"gpu_uuid":uuid,"driver_version":driver,"cuda_runtime":torch.version.cuda,"vram_gib":props.total_memory/2**30,"compute_capability":f"{props.major}.{props.minor}"}
(OUT/"gpu_info.json").write_text(json.dumps(info,indent=2));print(json.dumps(info,indent=2))


In [ ]:

S={"T4":(8.1,65,65,None,320),"L4":(30.3,121,242,242,300),"A100":(19.5,156,312,312,1555)}
k=next((x for x in S if x.lower() in gpu_name.lower()),None); fp32,tf32,fp16,bf16,bw=S.get(k,(None,)*5); print(gpu_name,k,bw)


In [ ]:

torch.backends.cuda.matmul.allow_tf32=True
sizes=[1024,4096,8192,16384]; types={"FP32":torch.float32,"TF32":torch.float32,"FP16":torch.float16,"BF16":torch.bfloat16}; reps={1024:20,4096:10,8192:3,16384:1}; peaks={"FP32":fp32,"TF32":tf32,"FP16":fp16,"BF16":bf16}
def runmm(n,p):
 a=torch.randn((n,n),device=device,dtype=types[p]);b=torch.randn_like(a)
 for _ in range(2):_=a@b
 torch.cuda.synchronize();s=torch.cuda.Event(True);e=torch.cuda.Event(True);s.record()
 for _ in range(reps[n]):_=a@b
 e.record();e.synchronize();sec=s.elapsed_time(e)/1000/reps[n];v=2*n**3/sec/1e12;del a,b;torch.cuda.empty_cache();return sec,v
rows=[]
for n in sizes:
 for p in types:
  try:
   sec,v=runmm(n,p);q=peaks[p];rows.append({"uuid":uuid,"gpu":gpu_name,"N":n,"precision":p,"seconds":sec,"repetitions":reps[n],"TFLOPS":v,"peak":q,"percent":100*v/q if q else np.nan,"status":"success"});print(p,n,v)
  except RuntimeError as x:
   if "out of memory" not in str(x).lower():raise
   torch.cuda.empty_cache();gc.collect();rows.append({"uuid":uuid,"gpu":gpu_name,"N":n,"precision":p,"status":"OOM"});print("OOM",p,n)
matmul=pd.DataFrame(rows);matmul.to_csv(OUT/"matmul_metrics.csv",index=False);display(matmul)
plt.figure(figsize=(9,5))
for p,g in matmul[matmul.status=="success"].groupby("precision"):plt.plot(g.N,g.TFLOPS,"o-",label=p)
plt.xscale("log",base=2);plt.xlabel("N");plt.ylabel("TFLOPS");plt.legend();plt.grid();plt.savefig(OUT/"matmul_tflops.png",dpi=180);plt.show()


In [ ]:

n=64*1024*1024;a=torch.randn(n,device=device);b=torch.randn_like(a)
for _ in range(3):c=a+b
torch.cuda.synchronize();s=torch.cuda.Event(True);e=torch.cuda.Event(True);s.record()
for _ in range(20):c=a+b
e.record();e.synchronize();sec=s.elapsed_time(e)/1000/20;gbs=3*n*4/sec/1e9
band={"uuid":uuid,"gpu":gpu_name,"effective_bandwidth_gbs":gbs,"specified_bandwidth_gbs":bw,"percent":100*gbs/bw if bw else np.nan,"arithmetic_intensity":1/12}
(OUT/"bandwidth_metrics.json").write_text(json.dumps(band,indent=2));print(band);del a,b,c;torch.cuda.empty_cache()


In [ ]:

B,H,D=1,1,64;Ls=[512,1024,2048,4096,8192,16384]
def test(L,fused):
 q=torch.randn((B,H,L,D),device=device,dtype=torch.float16);k=torch.randn_like(q);v=torch.randn_like(q);torch.cuda.reset_peak_memory_stats()
 for _ in range(2):
  if fused:z=torch.nn.functional.scaled_dot_product_attention(q,k,v)
  else:scores=q@k.transpose(-2,-1)/math.sqrt(D);probs=torch.softmax(scores,-1);z=probs@v
 torch.cuda.synchronize();s=torch.cuda.Event(True);e=torch.cuda.Event(True);s.record()
 for _ in range(2):
  if fused:z=torch.nn.functional.scaled_dot_product_attention(q,k,v)
  else:scores=q@k.transpose(-2,-1)/math.sqrt(D);probs=torch.softmax(scores,-1);z=probs@v
 e.record();e.synchronize();ms=s.elapsed_time(e)/2;mem=torch.cuda.max_memory_allocated()/2**20;del q,k,v,z;torch.cuda.empty_cache();gc.collect();return ms,mem
def sweep(fused):
 out=[];name="fused" if fused else "naive"
 for L in Ls:
  try:
   ms,mem=test(L,fused);out.append({"uuid":uuid,"gpu":gpu_name,"implementation":name,"L":L,"latency_ms":ms,"peak_memory_mib":mem,"status":"success"});print(name,L,ms,mem)
  except RuntimeError as x:
   if "out of memory" not in str(x).lower():raise
   torch.cuda.empty_cache();gc.collect();out.append({"uuid":uuid,"gpu":gpu_name,"implementation":name,"L":L,"status":"OOM"});print("OOM",name,L)
 return pd.DataFrame(out)
naive=sweep(False);fused=sweep(True);attention=pd.concat([naive,fused]);attention.to_csv(OUT/"attention_metrics.csv",index=False);display(attention)
for col,file in [("peak_memory_mib","attention_memory.png"),("latency_ms","attention_latency.png")]:
 plt.figure(figsize=(9,5))
 for name,g in attention[attention.status=="success"].groupby("implementation"):plt.plot(g.L,g[col],"o-",label=name)
 plt.xlabel("Sequence length");plt.ylabel(col);plt.legend();plt.grid();plt.savefig(OUT/file,dpi=180);plt.show()


In [ ]:

def bound(x):
 a=x[x.status=="success"].L.tolist();b=x[x.status=="OOM"].L.tolist();return max(a) if a else None,min(b) if b else None
f=naive[naive.status=="success"].dropna();z={"naive_boundary":bound(naive),"fused_boundary":bound(fused)}
if len(f)>=3:
 co=np.polyfit(f.L,f.peak_memory_mib,2);z.update({"quadratic_coefficient":float(co[0]),"linear_coefficient":float(co[1]),"intercept":float(co[2])})
(OUT/"attention_summary.json").write_text(json.dumps(z,indent=2));print(z)


## Thermal test

The next cell is disabled by default. Set RUN_THERMAL=True only when ready; it runs for 20 minutes.

In [ ]:

RUN_THERMAL=False
if RUN_THERMAL:
 start=time.time();records=[];duration=1200
 while time.time()-start<duration:
  a=torch.randn((4096,4096),device=device,dtype=torch.float16);b=torch.randn_like(a)
  torch.cuda.synchronize();t=time.perf_counter();_=a@b;torch.cuda.synchronize();v=2*4096**3/(time.perf_counter()-t)/1e12;del a,b;torch.cuda.empty_cache()
  q=cmd("nvidia-smi --query-gpu=timestamp,utilization.gpu,temperature.gpu,power.draw,clocks.sm,clocks.mem --format=csv,noheader,nounits").strip().split(",");records.append({"elapsed_seconds":time.time()-start,"throughput_tflops":v,"telemetry":q});time.sleep(5)
 pd.DataFrame(records).to_csv(OUT/"thermal_log.csv",index=False)
else:print("SKIPPED: change RUN_THERMAL to True for 20-minute test.")


In [ ]:

best=matmul[(matmul.precision=="BF16")&(matmul.status=="success")]
rows=[["Peak achieved TFLOPS (BF16)",best.TFLOPS.max() if len(best) else "not measured"],["Effective bandwidth (GB/s)",band["effective_bandwidth_gbs"]],["Naive OOM length",bound(naive)[1]],["Fused OOM length",bound(fused)[1]]]
rows += [["Thermal result","see thermal_log.csv" if (OUT/"thermal_log.csv").exists() else "not completed"]]
t=pd.DataFrame(rows,columns=["Measurement","Your GPU / Notes"]);t.to_csv(OUT/"summary_table.csv",index=False);(OUT/"METRICS.md").write_text(t.to_markdown(index=False));(OUT/"RUN_LOG.txt").write_text(f"GPU: {gpu_name}\nUUID: {uuid}\n");(OUT/"AI_USE.md").write_text("I used ChatGPT to help organize and explain the Colab benchmark notebook. I ran the measurements myself and reviewed the results.");display(t)
archive=shutil.make_archive("/content/gpu_assignment_outputs","zip",OUT);print(archive)
try:
 from google.colab import files;files.download(archive)
except:pass
